# PSO CUDA — notebook autonome (Kaggle ou Linux avec GPU NVIDIA)
Projet M2 *Massively Parallel PSO and DE algorithms* (UHA).

Exécuter les cellules dans l'ordre, sur Kaggle avec **Settings → Accelerator → GPU**.
Prérequis : GPU NVIDIA accessible, toolkit CUDA (`nvcc`), compilateur C++ (`g++`),
Python avec `pandas`, `numpy` et `matplotlib` (préinstallés sur Kaggle).
Un Mac sans GPU NVIDIA ne peut pas exécuter la partie CUDA.

Les sources embarquées sont la **seule source de vérité** : aucune copie depuis un dataset.
Elles sont générées dans `pso_gpu_work/src/`, les binaires dans `pso_gpu_work/bin/`
et les résultats dans `pso_gpu_work/results/`. Les fichiers DE et anciens CSV à la racine
ne sont ni compilés ni modifiés. Relancer le notebook remplace ses fichiers générés.

Protocole : quatre fonctions shifted ; D ∈ {10, 50, 100} ; P ∈ {50, 100, 500} ;
MaxFEs = 10⁴·D ; 10 runs. Paramètres inchangés : w = 0,9 → 0,4,
c1 = c2 = 2,0, vmax = 0,2 × largeur du domaine, précision double par défaut.

Le `pso_seq` embarqué est une **référence CPU facultative du même PSO**,
pas le PSO CPU de `project-cuda-de-pso.ipynb`. Les RNG CPU/GPU diffèrent :
une même graine ne garantit ni mêmes tirages ni mêmes résultats.


## 1. Configuration et vérification des prérequis


In [ ]:
from pathlib import Path
import csv
import hashlib
import importlib
import json
import math
import shutil
import subprocess
import sys

# Modifier ces options AVANT d'exécuter les cellules suivantes.
WORK_DIR = Path.cwd() / "pso_gpu_work"
RUN_CPU_REFERENCE = True     # False : compiler/exécuter seulement le GPU
USE_FLOAT = False            # True : état float, fitness toujours accumulée en double
CUDA_ARCH = None             # exemple : "sm_75" ; None = détection automatique
RUNS = 10
SEED = 1234
SHIFT = "rand"              # "zero", "rand" ou "file:/chemin/absolu"
# Échauffement GPU par configuration, dans le processus qui réalise les mesures.
WARMUP_RUNS = 3             # minimum de runs complets non comptabilisés
WARMUP_SECONDS = 2.0        # minimum de temps GPU cumulé, hors mesures
WARMUP_MAX_RUNS = 200       # plafond de sécurité ; avertissement si seuil non atteint

for package in ("pandas", "numpy", "matplotlib"):
    try:
        importlib.import_module(package)
    except ImportError as exc:
        raise RuntimeError(
            f"Dépendance Python manquante : {package}. Installer avec "
            f"{sys.executable} -m pip install pandas numpy matplotlib, puis relancer."
        ) from exc

for tool in ("nvidia-smi", "nvcc", "g++"):
    if shutil.which(tool) is None:
        raise RuntimeError(f"{tool} introuvable : activer le GPU Kaggle et installer le toolkit CUDA/C++.")

def checked(command, **kwargs):
    result = subprocess.run([str(x) for x in command], text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kwargs)
    print(result.stdout, end="")
    result.check_returncode()
    return result.stdout

GPU_INFO = checked(["nvidia-smi"])
NVCC_INFO = checked(["nvcc", "--version"])
CXX_INFO = checked(["g++", "--version"])
if CUDA_ARCH is None:
    try:
        caps = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            text=True).strip().splitlines()
        unique_caps = set(c.strip() for c in caps)
        if len(unique_caps) != 1:
            raise ValueError("GPU d'architectures différentes : sélectionner un GPU et préciser CUDA_ARCH")
        major, minor = unique_caps.pop().split(".")
        if not (major.isdigit() and minor.isdigit()):
            raise ValueError("compute capability non numérique")
        CUDA_ARCH = "sm_" + major + minor
    except (subprocess.SubprocessError, ValueError) as exc:
        raise RuntimeError("Détection impossible : renseigner CUDA_ARCH explicitement puis relancer.") from exc
if not CUDA_ARCH.startswith("sm_") or not CUDA_ARCH[3:].isdigit():
    raise ValueError("CUDA_ARCH doit avoir la forme sm_75, sm_80, etc.")

SRC_DIR, BIN_DIR, RESULTS_DIR = [WORK_DIR / p for p in ("src", "bin", "results")]
for directory in (SRC_DIR, BIN_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print("Architecture :", CUDA_ARCH, "— fichiers générés :", WORK_DIR)


## 2. Sources embarquées, isolées du DE
La cellule régénère uniquement les quatre sources dans `pso_gpu_work/src/`.
Les fichiers homonymes à la racine et les datasets Kaggle sont ignorés.
Les empreintes SHA-256 enregistrées permettent d'identifier le code exécuté.


In [ ]:
SOURCES = {}
SOURCES['kernel.h'] = r'''#pragma once
// ============================================================================
//  Massively parallel PSO (CUDA) - benchmark protocol of the M2 project
//  Objective functions : see bench_info() / fitness() below
// ============================================================================
#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <string>
#include <vector>

#ifdef __CUDACC__
  #define HD __host__ __device__
#else
  #define HD
#endif

// Precision of the swarm state. The fitness is ALWAYS accumulated in double
// (needed to reach errors ~1e-8 around biases such as -450).
// Compile with -DUSE_FLOAT for a pure single-precision swarm.
#ifdef USE_FLOAT
typedef float  real;
#else
typedef double real;
#endif

// ---------------------------------------------------------------------------
// Objective functions (same numbering as the original code)
//   0 : Levy                  (extra, not in the official protocol)
//   1 : Shifted Rastrigin     [-5, 5]^D        f* = -330
//   2 : Shifted Rosenbrock    [-100, 100]^D    f* =  390
//   3 : Shifted Griewank      [-600, 600]^D    f* = -180
//   4 : Shifted Sphere        [-100, 100]^D    f* = -450
// ---------------------------------------------------------------------------
enum ObjFunc { F_LEVY = 0, F_RASTRIGIN = 1, F_ROSENBROCK = 2, F_GRIEWANK = 3, F_SPHERE = 4 };

HD inline double f_bias(int func)
{
    switch (func) {
        case F_RASTRIGIN:  return -330.0;
        case F_ROSENBROCK: return  390.0;
        case F_GRIEWANK:   return -180.0;
        case F_SPHERE:     return -450.0;
        default:           return 0.0;
    }
}

// x[k * stride] is the k-th coordinate (stride = 1 on the host, stride = P on
// the GPU because the swarm is stored coordinate-major for coalesced access).
// o = shift vector (ignored by Levy).  One single implementation for CPU+GPU.
HD inline double fitness(const real* x, int stride, int D, int func, const real* o)
{
    const double PI = 3.14159265358979323846;
    double res = 0.0;

    switch (func) {
    case F_LEVY: {
        double w1 = 1.0 + ((double)x[0] - 1.0) / 4.0;
        double s  = sin(PI * w1);
        res = s * s;
        for (int i = 0; i < D - 1; i++) {
            double wi = 1.0 + ((double)x[i * stride] - 1.0) / 4.0;
            double t  = sin(PI * wi + 1.0);
            res += (wi - 1.0) * (wi - 1.0) * (1.0 + 10.0 * t * t);
        }
        double wd = 1.0 + ((double)x[(D - 1) * stride] - 1.0) / 4.0;
        double sd = sin(2.0 * PI * wd);
        res += (wd - 1.0) * (wd - 1.0) * (1.0 + sd * sd);
        break;
    }
    case F_RASTRIGIN: {
        for (int i = 0; i < D; i++) {
            double z = (double)x[i * stride] - (double)o[i];
            res += z * z - 10.0 * cos(2.0 * PI * z) + 10.0;
        }
        break;
    }
    case F_ROSENBROCK: {
        for (int i = 0; i < D - 1; i++) {
            double zi  = (double)x[i * stride]       - (double)o[i]     + 1.0;
            double zi1 = (double)x[(i + 1) * stride] - (double)o[i + 1] + 1.0;
            double a = zi * zi - zi1;
            double b = zi - 1.0;
            res += 100.0 * a * a + b * b;
        }
        break;
    }
    case F_GRIEWANK: {
        double sum = 0.0, prod = 1.0;          // prod MUST start at 1 (was 0 in v1)
        for (int i = 0; i < D; i++) {
            double z = (double)x[i * stride] - (double)o[i];
            sum  += z * z / 4000.0;
            prod *= cos(z / sqrt((double)(i + 1)));
        }
        res = sum - prod + 1.0;
        break;
    }
    case F_SPHERE: {
        for (int i = 0; i < D; i++) {
            double z = (double)x[i * stride] - (double)o[i];
            res += z * z;
        }
        break;
    }
    }
    return res + f_bias(func);
}

// ---------------------------------------------------------------------------
// Host-side helpers (kernel.cpp)
// ---------------------------------------------------------------------------
struct BenchInfo { const char* name; double lo, hi, bias; };
BenchInfo bench_info(int func);

// mode : "zero" | "rand" | "file:<path>"   (file = CEC2005 shift vector, first D values)
bool   make_shift(int func, int dim, const std::string& mode, std::vector<real>& o);
double host_fitness(const std::vector<real>& x, int func, const std::vector<real>& o);

// ---------------------------------------------------------------------------
// PSO (kernel.cu)
// ---------------------------------------------------------------------------
struct PSOParams {
    int       func   = F_SPHERE;
    int       dim    = 10;
    int       pop    = 50;                 // 2 .. 1024
    long long maxFE  = 100000;             // protocol : 10^4 * dim
    double    wMax   = 0.9, wMin = 0.4;    // inertia, linearly decreasing
    double    c1     = 2.0,  c2   = 2.0;
    double    vmaxFrac = 0.2;              // |v| <= vmaxFrac * (hi - lo)
    unsigned long long seed = 1234ULL;
};

struct PSOResult {
    double            bestFit = 0.0;
    double            timeMs  = 0.0;       // GPU time measured with cudaEvent
    long long         fe      = 0;         // function evaluations really used
    int               iters   = 0;
    std::vector<real> gBest;
};

PSOResult cuda_pso(const PSOParams& prm, const std::vector<real>& shift);   // kernel.cu (GPU)
PSOResult seq_pso (const PSOParams& prm, const std::vector<real>& shift);   // pso_seq.cpp (CPU, 1 thread)
'''
SOURCES['kernel.cu'] = r'''#include <cuda_runtime.h>
#include <curand_kernel.h>
#include "kernel.h"

#define CUDA_CHECK(call)                                                        \
    do {                                                                        \
        cudaError_t e_ = (call);                                                \
        if (e_ != cudaSuccess) {                                                \
            fprintf(stderr, "CUDA error %s:%d : %s\n", __FILE__, __LINE__,      \
                    cudaGetErrorString(e_));                                    \
            exit(1);                                                            \
        }                                                                       \
    } while (0)

#define INF_FIT 1.0e300
static const int TPB = 128;                       // threads per block

typedef curandStatePhilox4_32_10_t RngState;

// Swarm layout is coordinate-major (SoA): element (particle p, dim d) lives at
// d * P + p  ->  one thread per particle gives fully coalesced accesses.

__device__ __forceinline__ real urand(RngState* s)
{
#ifdef USE_FLOAT
    return curand_uniform(s);
#else
    return curand_uniform_double(s);
#endif
}

// ---- 1. initialisation : one thread per (particle, dimension) --------------
__global__ void k_init(real* pos, real* vel, real* pbest, double* pfit,
                       RngState* rng, int P, int D, real lo, real hi,
                       unsigned long long seed)
{
    int e = blockIdx.x * blockDim.x + threadIdx.x;
    if (e >= P * D) return;

    RngState st;
    curand_init(seed, (unsigned long long)e, 0ULL, &st);
    real x = lo + (hi - lo) * urand(&st);
    pos[e]   = x;
    pbest[e] = x;
    vel[e]   = (real)0;
    rng[e]   = st;
    if (e < P) pfit[e] = INF_FIT;
}

// ---- 2. velocity / position update : one thread per element ----------------
// independent r1, r2 for every particle AND every dimension (standard PSO)
__global__ void k_update(real* pos, real* vel, const real* pbest,
                         const real* gbest, RngState* rng, int P, int D,
                         real lo, real hi, real w, real c1, real c2, real vmax)
{
    int e = blockIdx.x * blockDim.x + threadIdx.x;
    if (e >= P * D) return;
    int d = e / P;

    RngState st = rng[e];
    real r1 = urand(&st);
    real r2 = urand(&st);
    rng[e] = st;

    real x = pos[e];
    real v = w * vel[e] + c1 * r1 * (pbest[e] - x) + c2 * r2 * (gbest[d] - x);
    v = fmax(-vmax, fmin(vmax, v));               // velocity clamping
    x += v;
    if (x < lo)      { x = lo; v = (real)0; }     // position clamping
    else if (x > hi) { x = hi; v = (real)0; }
    vel[e] = v;
    pos[e] = x;
}

// ---- 3. evaluation + pBest update : one thread per particle ----------------
// pBest fitness is stored (pfit) -> never recomputed.
__global__ void k_eval(const real* pos, real* pbest, double* pfit, int P, int D,
                       int func, const real* shift)
{
    int p = blockIdx.x * blockDim.x + threadIdx.x;
    if (p >= P) return;

    double f = fitness(pos + p, P, D, func, shift);
    if (f < pfit[p]) {
        pfit[p] = f;
        for (int d = 0; d < D; d++) pbest[(size_t)d * P + p] = pos[(size_t)d * P + p];
    }
}

// ---- 4. gBest : argmin reduction over pBest fitness, entirely on the GPU ----
// launched with ONE block whose size is a power of two >= P (P <= 1024)
__global__ void k_gbest(const real* pbest, const double* pfit, real* gbest,
                        double* gfit, int P, int D)
{
    __shared__ double sv[1024];
    __shared__ int    si[1024];
    int t = threadIdx.x;

    sv[t] = (t < P) ? pfit[t] : INF_FIT;
    si[t] = t;
    __syncthreads();

    for (int s = blockDim.x >> 1; s > 0; s >>= 1) {
        if (t < s && sv[t + s] < sv[t]) { sv[t] = sv[t + s]; si[t] = si[t + s]; }
        __syncthreads();
    }

    const double best = sv[0];
    const int    bi   = si[0];
    const double old  = *gfit;       // read by everybody BEFORE thread 0 writes
    __syncthreads();

    if (best < old) {
        for (int d = t; d < D; d += blockDim.x) gbest[d] = pbest[(size_t)d * P + bi];
        if (t == 0) *gfit = best;
    }
}

// ---------------------------------------------------------------------------
PSOResult cuda_pso(const PSOParams& prm, const std::vector<real>& shift)
{
    const int P = prm.pop, D = prm.dim;
    if (P < 2 || P > 1024 || D < 1 || (int)shift.size() < D) {
        fprintf(stderr, "cuda_pso: invalid parameters (pop in [2,1024], dim>=1)\n");
        exit(1);
    }
    const BenchInfo bi = bench_info(prm.func);
    const size_t n = (size_t)P * D;

    static bool warm = false;
    if (!warm) { CUDA_CHECK(cudaFree(0)); warm = true; }   // create context

    real *d_pos, *d_vel, *d_pbest, *d_gbest, *d_shift;
    double *d_pfit, *d_gfit;
    RngState* d_rng;
    CUDA_CHECK(cudaMalloc(&d_pos,   n * sizeof(real)));
    CUDA_CHECK(cudaMalloc(&d_vel,   n * sizeof(real)));
    CUDA_CHECK(cudaMalloc(&d_pbest, n * sizeof(real)));
    CUDA_CHECK(cudaMalloc(&d_gbest, D * sizeof(real)));
    CUDA_CHECK(cudaMalloc(&d_shift, D * sizeof(real)));
    CUDA_CHECK(cudaMalloc(&d_pfit,  P * sizeof(double)));
    CUDA_CHECK(cudaMalloc(&d_gfit,  sizeof(double)));
    CUDA_CHECK(cudaMalloc(&d_rng,   n * sizeof(RngState)));

    CUDA_CHECK(cudaMemcpy(d_shift, shift.data(), D * sizeof(real), cudaMemcpyHostToDevice));
    const double inf = INF_FIT;
    CUDA_CHECK(cudaMemcpy(d_gfit, &inf, sizeof(double), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_gbest, 0, D * sizeof(real)));

    const int blocksE = (int)((n + TPB - 1) / TPB);
    const int blocksP = (P + TPB - 1) / TPB;
    int gbThreads = 32;
    while (gbThreads < P) gbThreads <<= 1;

    const real lo   = (real)bi.lo, hi = (real)bi.hi;
    const real vmax = (real)(prm.vmaxFrac * (bi.hi - bi.lo));

    // FE budget : initial evaluation (P) + nIter * P  <=  maxFE
    long long nIter = prm.maxFE / P - 1;
    if (nIter < 0) nIter = 0;

    cudaEvent_t e0, e1;
    CUDA_CHECK(cudaEventCreate(&e0));
    CUDA_CHECK(cudaEventCreate(&e1));
    CUDA_CHECK(cudaEventRecord(e0));

    k_init<<<blocksE, TPB>>>(d_pos, d_vel, d_pbest, d_pfit, d_rng, P, D, lo, hi, prm.seed);
    k_eval<<<blocksP, TPB>>>(d_pos, d_pbest, d_pfit, P, D, prm.func, d_shift);
    k_gbest<<<1, gbThreads>>>(d_pbest, d_pfit, d_gbest, d_gfit, P, D);

    for (long long it = 0; it < nIter; it++) {
        double w = (nIter > 1) ? prm.wMax - (prm.wMax - prm.wMin) * (double)it / (double)(nIter - 1)
                               : prm.wMax;
        k_update<<<blocksE, TPB>>>(d_pos, d_vel, d_pbest, d_gbest, d_rng, P, D, lo, hi,
                                   (real)w, (real)prm.c1, (real)prm.c2, vmax);
        k_eval<<<blocksP, TPB>>>(d_pos, d_pbest, d_pfit, P, D, prm.func, d_shift);
        k_gbest<<<1, gbThreads>>>(d_pbest, d_pfit, d_gbest, d_gfit, P, D);
    }
    CUDA_CHECK(cudaGetLastError());

    CUDA_CHECK(cudaEventRecord(e1));
    CUDA_CHECK(cudaEventSynchronize(e1));        // also surfaces async kernel errors
    float ms = 0.f;
    CUDA_CHECK(cudaEventElapsedTime(&ms, e0, e1));

    PSOResult r;
    r.gBest.resize(D);
    CUDA_CHECK(cudaMemcpy(r.gBest.data(), d_gbest, D * sizeof(real), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(&r.bestFit, d_gfit, sizeof(double), cudaMemcpyDeviceToHost));
    r.timeMs = ms;
    r.iters  = (int)nIter;
    r.fe     = (long long)P * (1 + nIter);

    cudaEventDestroy(e0); cudaEventDestroy(e1);
    cudaFree(d_pos); cudaFree(d_vel); cudaFree(d_pbest); cudaFree(d_gbest);
    cudaFree(d_shift); cudaFree(d_pfit); cudaFree(d_gfit); cudaFree(d_rng);
    return r;
}
'''
SOURCES['kernel.cpp'] = r'''#include "kernel.h"
#include <chrono>
#include <fstream>
#include <random>

BenchInfo bench_info(int func)
{
    switch (func) {
        case F_LEVY:       return { "Levy",       -10.0,   10.0,    0.0 };
        case F_RASTRIGIN:  return { "Rastrigin",   -5.0,    5.0, -330.0 };
        case F_ROSENBROCK: return { "Rosenbrock", -100.0,  100.0,  390.0 };
        case F_GRIEWANK:   return { "Griewank",   -600.0,  600.0, -180.0 };
        case F_SPHERE:     return { "Sphere",     -100.0,  100.0, -450.0 };
        default:           return { "?", 0, 0, 0 };
    }
}

// Shift vector o :  z = x - o, optimum at x* = o, f(x*) = bias.
//  "zero"       : o = 0 (behaviour of the original code)
//  "rand"       : reproducible o_i in [0.8*lo, 0.8*hi] (fixed seed, prefix-stable in dim)
//  "file:<path>": first `dim` numbers of a CEC2005 data file (e.g. sphere_func_data.txt)
bool make_shift(int func, int dim, const std::string& mode, std::vector<real>& o)
{
    o.assign(dim, (real)0);
    if (func == F_LEVY || mode == "zero") return true;

    BenchInfo b = bench_info(func);
    if (mode == "rand") {
        std::mt19937_64 gen(20261027ULL + (unsigned)func);
        std::uniform_real_distribution<double> U(0.8 * b.lo, 0.8 * b.hi);
        for (int i = 0; i < dim; i++) o[i] = (real)U(gen);
        return true;
    }
    if (mode.rfind("file:", 0) == 0) {
        std::ifstream f(mode.substr(5));
        if (!f) { fprintf(stderr, "cannot open shift file %s\n", mode.c_str() + 5); return false; }
        double v;
        for (int i = 0; i < dim; i++) {
            if (!(f >> v)) { fprintf(stderr, "shift file has fewer than %d values\n", dim); return false; }
            o[i] = (real)v;
        }
        return true;
    }
    fprintf(stderr, "unknown shift mode '%s'\n", mode.c_str());
    return false;
}

// Same code as the GPU one (kernel.h) -> used to double-check the GPU result.
double host_fitness(const std::vector<real>& x, int func, const std::vector<real>& o)
{
    return fitness(x.data(), 1, (int)x.size(), func, o.data());
}


// ============================================================================
//  Sequential PSO (CPU, 1 thread) - reference implementation, compiled with the
//  rest of kernel.cpp (build with -DSEQUENTIAL to use it from main.cpp).
//  Same algorithm, parameters, budget and output as the CUDA version (kernel.cu):
//    - synchronous PSO : gBest is updated once per generation, after all pBest
//    - independent r1, r2 per particle AND per dimension
//    - inertia decreasing linearly wMax -> wMin, velocity and position clamping
//    - FEs = P * (1 + nIter) <= MaxFEs
//  Only the RNG differs (mt19937_64 here, cuRAND Philox on the GPU): results are
//  not bit-identical; statistical agreement must be assessed experimentally.
//  Memory layout is particle-major (p*D + d), the cache-friendly one on a CPU.
// ============================================================================
PSOResult seq_pso(const PSOParams& prm, const std::vector<real>& shift)
{
    const int P = prm.pop, D = prm.dim;
    if (P < 2 || D < 1 || (int)shift.size() < D) {
        fprintf(stderr, "seq_pso: invalid parameters (pop >= 2, dim >= 1)\n");
        exit(1);
    }
    const BenchInfo bi = bench_info(prm.func);
    const size_t n = (size_t)P * D;
    const real lo   = (real)bi.lo, hi = (real)bi.hi;
    const real vmax = (real)(prm.vmaxFrac * (bi.hi - bi.lo));
    const real c1 = (real)prm.c1, c2 = (real)prm.c2;

    long long nIter = prm.maxFE / P - 1;           // initial evaluation costs P FEs
    if (nIter < 0) nIter = 0;

    std::mt19937_64 gen(prm.seed);
    std::uniform_real_distribution<real> U01((real)0, (real)1);

    std::vector<real>   pos(n), vel(n, (real)0), pbest(n), gbest(D, (real)0);
    std::vector<double> pfit(P, 1.0e300);
    double gfit = 1.0e300;

    // evaluation + pBest update (kernel E on the GPU)
    auto evaluate = [&]() {
        for (int p = 0; p < P; p++) {
            double f = fitness(&pos[(size_t)p * D], 1, D, prm.func, shift.data());
            if (f < pfit[p]) {
                pfit[p] = f;
                for (int d = 0; d < D; d++) pbest[(size_t)p * D + d] = pos[(size_t)p * D + d];
            }
        }
    };
    // gBest = argmin of pBest fitness (kernel G on the GPU)
    auto reduce_gbest = [&]() {
        int b = 0;
        for (int p = 1; p < P; p++) if (pfit[p] < pfit[b]) b = p;
        if (pfit[b] < gfit) {
            gfit = pfit[b];
            for (int d = 0; d < D; d++) gbest[d] = pbest[(size_t)b * D + d];
        }
    };

    auto t0 = std::chrono::steady_clock::now();

    // initialisation (kernel I)
    for (size_t e = 0; e < n; e++) {
        pos[e]   = lo + (hi - lo) * U01(gen);
        pbest[e] = pos[e];
    }
    evaluate();
    reduce_gbest();

    // main loop
    for (long long it = 0; it < nIter; it++) {
        const real w = (real)((nIter > 1)
            ? prm.wMax - (prm.wMax - prm.wMin) * (double)it / (double)(nIter - 1)
            : prm.wMax);

        // velocity & position update (kernel U)
        for (int p = 0; p < P; p++) {
            for (int d = 0; d < D; d++) {
                const size_t e = (size_t)p * D + d;
                const real r1 = U01(gen), r2 = U01(gen);
                real x = pos[e];
                real v = w * vel[e] + c1 * r1 * (pbest[e] - x) + c2 * r2 * (gbest[d] - x);
                if (v >  vmax) v =  vmax;
                if (v < -vmax) v = -vmax;
                x += v;
                if (x < lo)      { x = lo; v = (real)0; }
                else if (x > hi) { x = hi; v = (real)0; }
                vel[e] = v;
                pos[e] = x;
            }
        }
        evaluate();
        reduce_gbest();
    }

    auto t1 = std::chrono::steady_clock::now();

    PSOResult r;
    r.bestFit = gfit;
    r.gBest   = gbest;
    r.timeMs  = std::chrono::duration<double, std::milli>(t1 - t0).count();
    r.iters   = (int)nIter;
    r.fe      = (long long)P * (1 + nIter);
    return r;
}
'''
SOURCES['main.cpp'] = r'''#include "kernel.h"
#include <algorithm>
#include <cmath>
#include <cstring>
#include <cerrno>
#include <climits>

#ifdef SEQUENTIAL
  #define RUN_PSO seq_pso
  static const char* IMPL = "sequential CPU (1 thread)";
#else
  #define RUN_PSO cuda_pso
  static const char* IMPL = "CUDA GPU";
#endif

static void usage(const char* p)
{
    printf(
        "Usage: %s [options]\n"
        "  --func  all | 0..4      0 Levy, 1 Rastrigin, 2 Rosenbrock, 3 Griewank, 4 Sphere\n"
        "                          (all = the 4 official benchmarks 4,1,2,3)   [all]\n"
        "  --dim   D               problem dimension                            [10]\n"
        "  --pop   P               population size (2..1024)                    [50]\n"
        "  --runs  R               independent runs                             [10]\n"
        "  --seed  S               base seed (run k uses S+k)                   [1234]\n"
        "  --shift zero|rand|file:PATH   shift vector o                         [rand]\n"
        "  --sweep                 official protocol: dim {10,50,100} x pop {50,100,500}\n"
        "  --out   FILE.csv        write every run (for Wilcoxon / Kruskal-Wallis)\n"
        "  --warmup-runs N         GPU warm-up: minimum full runs per configuration [3]\n"
        "  --warmup-seconds S      GPU warm-up: minimum cumulative GPU seconds     [2]\n"
        "  --warmup-max-runs N     GPU warm-up: safety cap per configuration      [200]\n"
        "  --warmup-out FILE.csv   separate GPU warm-up timings (never measured runs)\n"
        "Warm-up is GPU-only. Disable with --warmup-runs 0 --warmup-seconds 0.\n"
        "MaxFEs = 10^4 * dim.\n", p);
}

static int warmup_count(const std::string& value)
{
    char* end = nullptr; errno = 0;
    long result = strtol(value.c_str(), &end, 10);
    if (errno || end == value.c_str() || *end || result < 0 || result > INT_MAX) {
        fprintf(stderr, "warm-up counts must be nonnegative integers\n"); exit(1);
    }
    return (int)result;
}

static double warmup_seconds(const std::string& value)
{
    char* end = nullptr; errno = 0;
    double result = strtod(value.c_str(), &end);
    if (errno || end == value.c_str() || *end || !std::isfinite(result) || result < 0) {
        fprintf(stderr, "--warmup-seconds must be finite and nonnegative\n"); exit(1);
    }
    return result;
}

int main(int argc, char** argv)
{
    std::vector<int> funcs = { F_SPHERE, F_RASTRIGIN, F_ROSENBROCK, F_GRIEWANK };
    std::vector<int> dims = { 10 }, pops = { 50 };
    int runs = 10;
    unsigned long long seed = 1234ULL;
    std::string shiftMode = "rand";
    std::string outPath;
    bool fileShift = false;
    int warmupRuns = 3, warmupMaxRuns = 200;
    double warmupSeconds = 2.0;
    std::string warmupPath;

    for (int a = 1; a < argc; a++) {
        std::string s = argv[a];
        auto next = [&]() -> std::string {
            if (a + 1 >= argc) { usage(argv[0]); exit(1); }
            return argv[++a];
        };
        if (s == "--func") {
            std::string v = next();
            if (v != "all") funcs = { atoi(v.c_str()) };
        } else if (s == "--dim")   dims = { atoi(next().c_str()) };
        else if (s == "--pop")     pops = { atoi(next().c_str()) };
        else if (s == "--runs")    runs = atoi(next().c_str());
        else if (s == "--seed")    seed = strtoull(next().c_str(), nullptr, 10);
        else if (s == "--shift")   { shiftMode = next(); fileShift = shiftMode.rfind("file:", 0) == 0; }
        else if (s == "--sweep")   { dims = { 10, 50, 100 }; pops = { 50, 100, 500 }; }
        else if (s == "--out")     outPath = next();
        else if (s == "--warmup-runs") warmupRuns = warmup_count(next());
        else if (s == "--warmup-seconds") warmupSeconds = warmup_seconds(next());
        else if (s == "--warmup-max-runs") warmupMaxRuns = warmup_count(next());
        else if (s == "--warmup-out") warmupPath = next();
        else { usage(argv[0]); return 1; }
    }
    if (runs < 1) { fprintf(stderr, "--runs must be >= 1\n"); return 1; }
    if (warmupMaxRuns < 1 || warmupRuns > warmupMaxRuns) {
        fprintf(stderr, "warm-up requires max-runs >= 1 and minimum runs <= max-runs\n"); return 1;
    }
    if (!warmupPath.empty() && warmupPath == outPath) {
        fprintf(stderr, "warm-up and measured CSV paths must differ\n"); return 1;
    }
#ifdef SEQUENTIAL
    (void)warmupSeconds;  // GPU-only setting; CPU measurement protocol is unchanged.
    if (!warmupPath.empty()) { fprintf(stderr, "--warmup-out is GPU-only\n"); return 1; }
#endif
    if (fileShift && funcs.size() != 1) {
        fprintf(stderr, "--shift file:... requires a single --func\n");
        return 1;
    }
    for (int f : funcs) if (f < 0 || f > 4) { fprintf(stderr, "bad --func\n"); return 1; }
    for (int p : pops)  if (p < 2 || p > 1024) { fprintf(stderr, "--pop must be in [2,1024]\n"); return 1; }
    for (int d : dims)  if (d < 1) { fprintf(stderr, "--dim must be >= 1\n"); return 1; }

    FILE* csv = nullptr;
    if (!outPath.empty()) {
        csv = fopen(outPath.c_str(), "w");
        if (!csv) { perror("open --out"); return 1; }
        fprintf(csv, "func,dim,pop,run,seed,fe,best,error,time_ms\n");
    }

    FILE* warmCsv = nullptr;
    if (!warmupPath.empty()) {
        warmCsv = fopen(warmupPath.c_str(), "w");
        if (!warmCsv) { perror("open --warmup-out"); if (csv) fclose(csv); return 1; }
        fprintf(warmCsv, "func,dim,pop,warmup_run,seed,fe,time_ms\n");
    }

    printf("Implementation : %s\n", IMPL);
    printf("%-10s %4s %4s %4s | %-14s %-14s %-14s %-14s | %10s\n",
           "function", "dim", "pop", "runs", "mean", "std", "best", "mean_error", "time_ms");

    int mismatches = 0;
    for (int func : funcs) {
        BenchInfo bi = bench_info(func);
        for (int dim : dims) {
            std::vector<real> o;
            if (!make_shift(func, dim, shiftMode, o)) return 1;
            for (int pop : pops) {
                PSOParams prm;
                prm.func = func; prm.dim = dim; prm.pop = pop;
                prm.maxFE = 10000LL * dim;                       // protocol

#ifndef SEQUENTIAL
                // Same workload and CUDA context as the measured runs below.
                // Each call starts a fresh swarm; no warm-up solution is retained.
                int warmed = 0;
                double warmMs = 0.0;
                while (warmed < warmupMaxRuns &&
                       (warmed < warmupRuns || warmMs / 1000.0 < warmupSeconds)) {
                    prm.seed = seed + (unsigned long long)warmed;
                    PSOResult warm = cuda_pso(prm, o);
                    double hf = host_fitness(warm.gBest, func, o);
                    if (!std::isfinite(hf) || !std::isfinite(warm.bestFit) ||
                        !std::isfinite(warm.timeMs) || warm.timeMs <= 0 ||
                        std::fabs(hf - warm.bestFit) > 1e-6 * (1.0 + std::fabs(hf))) {
                        fprintf(stderr, "ERROR: invalid warm-up result\n");
                        if (warmCsv) fclose(warmCsv);
                        if (csv) fclose(csv);
                        return 1;
                    }
                    if (warmCsv) fprintf(warmCsv, "%s,%d,%d,%d,%llu,%lld,%.6f\n",
                        bi.name, dim, pop, warmed, prm.seed, warm.fe, warm.timeMs);
                    warmMs += warm.timeMs;
                    ++warmed;
                }
                const bool targetReached = warmed >= warmupRuns && warmMs / 1000.0 >= warmupSeconds;
                printf("WARMUP %s D=%d P=%d: %d runs, %.3f GPU s; %s (excluded from measurements)\n",
                    bi.name, dim, pop, warmed, warmMs / 1000.0,
                    targetReached ? "duration/count target reached, stability not guaranteed" : "WARNING: safety cap reached before target");
                fflush(stdout);
#endif

                std::vector<double> fit(runs);
                double tsum = 0.0;
                for (int r = 0; r < runs; r++) {
                    prm.seed = seed + (unsigned long long)r;
                    PSOResult res = RUN_PSO(prm, o);
                    fit[r] = res.bestFit;
                    tsum  += res.timeMs;

                    // consistency check GPU value <-> host recomputation
                    double hf = host_fitness(res.gBest, func, o);
                    if (!std::isfinite(hf) || !std::isfinite(res.bestFit) ||
                        std::fabs(hf - res.bestFit) > 1e-6 * (1.0 + std::fabs(hf))) mismatches++;

                    if (csv)
                        fprintf(csv, "%s,%d,%d,%d,%llu,%lld,%.12e,%.12e,%.3f\n", bi.name, dim, pop, r,
                                prm.seed, res.fe, res.bestFit, res.bestFit - bi.bias, res.timeMs);
                }
                double mean = 0.0;
                for (double v : fit) mean += v;
                mean /= runs;
                double var = 0.0;
                for (double v : fit) var += (v - mean) * (v - mean);
                double sd = runs > 1 ? std::sqrt(var / (runs - 1)) : 0.0;   // sample std
                double best = *std::min_element(fit.begin(), fit.end());

                printf("%-10s %4d %4d %4d | %-14.6e %-14.6e %-14.6e %-14.6e | %10.2f\n",
                       bi.name, dim, pop, runs, mean, sd, best, mean - bi.bias, tsum / runs);
                fflush(stdout);
            }
        }
    }
    if (csv) fclose(csv);
    if (warmCsv) fclose(warmCsv);
    if (mismatches) fprintf(stderr, "ERROR: %d run(s) with GPU/host fitness mismatch\n", mismatches);
    return mismatches ? 1 : 0;
}
'''


NAMES = ("kernel.h", "kernel.cu", "kernel.cpp", "main.cpp")
assert set(NAMES) == set(SOURCES), "Noms des sources incohérents"
SOURCE_HASHES = {}
for name in NAMES:
    content = SOURCES[name].encode("utf-8")
    (SRC_DIR / name).write_bytes(content)
    SOURCE_HASHES[name] = hashlib.sha256(content).hexdigest()
    print(f"{name}: écrit dans {SRC_DIR}, SHA-256 {SOURCE_HASHES[name][:12]}")
(SRC_DIR / "sources.sha256.json").write_text(json.dumps(SOURCE_HASHES, indent=2))


## 3. Compilation et lanceur contrôlé
GPU et référence CPU compilent les mêmes sources isolées en C++14 et avec la même précision.
Toute erreur de compilation ou d'exécution arrête la cellule (pas de pipeline `tee`
qui masquerait le code de retour). Les sorties des sweeps sont également enregistrées.


In [ ]:
# Invalider le build précédent avant toute nouvelle tentative de compilation.
BUILD_CONFIG = None
precision_flags = ["-DUSE_FLOAT"] if USE_FLOAT else []
GPU_EXE, CPU_EXE = BIN_DIR / "pso", BIN_DIR / "pso_seq"
BUILD_COMMANDS = [
    ["nvcc", "-O3", "-std=c++14", f"-arch={CUDA_ARCH}", *precision_flags,
     "main.cpp", "kernel.cpp", "kernel.cu", "-o", str(GPU_EXE)],
]
if RUN_CPU_REFERENCE:
    BUILD_COMMANDS.append(
        ["g++", "-O3", "-std=c++14", "-DSEQUENTIAL", *precision_flags,
         "main.cpp", "kernel.cpp", "-o", str(CPU_EXE)])
for command in BUILD_COMMANDS:
    checked(command, cwd=SRC_DIR)

# Détecter une modification des sources/configurations après compilation.
BUILD_HASHES = dict(SOURCE_HASHES)
BUILD_CONFIG = (CUDA_ARCH, USE_FLOAT, RUN_CPU_REFERENCE)

def run_pso(impl, args, output_name=None, expected_rows=None, *,
            warmup_runs=None, warmup_seconds=None, warmup_max_runs=None):
    if (CUDA_ARCH, USE_FLOAT, RUN_CPU_REFERENCE) != BUILD_CONFIG:
        raise RuntimeError("Configuration modifiée : recompiler avant l'exécution.")
    for name, digest in BUILD_HASHES.items():
        if hashlib.sha256((SRC_DIR / name).read_bytes()).hexdigest() != digest:
            raise RuntimeError("Sources modifiées : recompiler avant l'exécution.")
    if impl not in ("gpu", "cpu") or (impl == "cpu" and not RUN_CPU_REFERENCE):
        raise ValueError("Implémentation indisponible")
    command = [str(GPU_EXE if impl == "gpu" else CPU_EXE), *map(str, args)]
    warmup = dict(applied=False)
    if impl == "gpu":
        minimum = WARMUP_RUNS if warmup_runs is None else warmup_runs
        seconds = WARMUP_SECONDS if warmup_seconds is None else warmup_seconds
        maximum = WARMUP_MAX_RUNS if warmup_max_runs is None else warmup_max_runs
        if (not isinstance(minimum, int) or not isinstance(maximum, int) or
                minimum < 0 or maximum < 1 or minimum > maximum or
                not math.isfinite(seconds) or seconds < 0):
            raise ValueError("Paramètres de warm-up invalides")
        if any(str(arg).startswith("--warmup-") for arg in args):
            raise ValueError("Utiliser les paramètres warmup_* de run_pso, pas les arguments CLI")
        command += ["--warmup-runs", str(minimum), "--warmup-seconds", str(seconds),
                    "--warmup-max-runs", str(maximum)]
        warmup = dict(applied=minimum > 0 or seconds > 0, min_runs=minimum,
                      min_gpu_seconds=seconds, max_runs=maximum,
                      scope="same process, function, dimension, population, shift and FE budget",
                      stopping_rule="minimum count AND cumulative GPU time, bounded by max_runs",
                      stability_guaranteed=False)
    result_path = None
    warmup_path = None
    if output_name:
        result_path = RESULTS_DIR / output_name
        # Une ancienne sortie ne doit pas survivre à un nouveau run raté.
        result_path.unlink(missing_ok=True)
        result_path.with_suffix(".json").unlink(missing_ok=True)
        command += ["--out", str(result_path)]
        if impl == "gpu":
            warmup_path = result_path.with_name(result_path.stem + "_warmup.csv")
            warmup_path.unlink(missing_ok=True)
            command += ["--warmup-out", str(warmup_path)]
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout, end="")
    if output_name:
        (RESULTS_DIR / (Path(output_name).stem + ".txt")).write_text(result.stdout)
    if result.returncode:
        if result_path:
            result_path.unlink(missing_ok=True)
        if warmup_path:
            warmup_path.unlink(missing_ok=True)
        result.check_returncode()
    if result_path:
        try:
            with result_path.open(newline="") as f:
                reader = csv.DictReader(f)
                if reader.fieldnames != ["func", "dim", "pop", "run", "seed", "fe", "best", "error", "time_ms"]:
                    raise ValueError("Schéma CSV inattendu")
                rows = list(reader)
            if not rows or (expected_rows is not None and len(rows) != expected_rows):
                raise ValueError(f"Nombre de runs inattendu : {len(rows)}, attendu : {expected_rows}")
            keys = set()
            for row in rows:
                key = tuple(row[k] for k in ("func", "dim", "pop", "run"))
                if key in keys:
                    raise ValueError(f"Run dupliqué : {key}")
                keys.add(key)
                d, p, fe = (int(row[k]) for k in ("dim", "pop", "fe"))
                if fe != (10000 * d // p) * p or fe > 10000 * d:
                    raise ValueError("Budget d'évaluations incohérent")
                if not all(math.isfinite(float(row[k])) for k in ("best", "error", "time_ms")):
                    raise ValueError("Résultat non fini")
                if float(row["time_ms"]) <= 0:
                    raise ValueError("Temps non positif")
            if warmup_path:
                with warmup_path.open(newline="") as f:
                    reader = csv.DictReader(f)
                    if reader.fieldnames != ["func", "dim", "pop", "warmup_run", "seed", "fe", "time_ms"]:
                        raise ValueError("Schéma du CSV warm-up inattendu")
                    warm_rows = list(reader)
                groups = {}
                for row in warm_rows:
                    key = tuple(row[k] for k in ("func", "dim", "pop"))
                    groups.setdefault(key, []).append(row)
                    t = float(row["time_ms"])
                    d, p = int(row["dim"]), int(row["pop"])
                    if not math.isfinite(t) or t <= 0 or int(row["fe"]) != (10000 * d // p) * p:
                        raise ValueError("Run de warm-up invalide")
                expected_groups = {tuple(r[k] for k in ("func", "dim", "pop")) for r in rows}
                if set(groups) != (expected_groups if warmup["applied"] else set()):
                    raise ValueError("Configurations du warm-up incohérentes")
                report = []
                for key, group in groups.items():
                    count = len(group)
                    if not minimum <= count <= maximum or [int(r["warmup_run"]) for r in group] != list(range(count)):
                        raise ValueError("Nombre ou indices de warm-up incohérents")
                    gpu_seconds = sum(float(r["time_ms"]) for r in group) / 1000.0
                    reached = gpu_seconds + count * 0.0000000005 >= seconds
                    if not reached and count != maximum:
                        raise ValueError("Warm-up interrompu avant le seuil et le plafond")
                    report.append(dict(func=key[0], dim=int(key[1]), pop=int(key[2]),
                                       runs=count, gpu_seconds=gpu_seconds, target_reached=reached))
                warmup.update(csv_file=warmup_path.name,
                              csv_sha256=hashlib.sha256(warmup_path.read_bytes()).hexdigest(),
                              configurations=report)
            metadata = dict(impl=f"PSO {impl}", command=command, source_sha256=BUILD_HASHES,
                            precision="float_state_double_fitness" if USE_FLOAT else "double",
                            cuda_arch=CUDA_ARCH, build_commands=BUILD_COMMANDS,
                            gpu_info=GPU_INFO, nvcc=NVCC_INFO, cxx=CXX_INFO,
                            shift_generator_seed="20261027 + func", w_max=0.9, w_min=0.4,
                            c1=2.0, c2=2.0, vmax_fraction=0.2,
                            timing="initialisation et boucle ; allocations et transferts exclus",
                            warmup=warmup,
                            csv_sha256=hashlib.sha256(result_path.read_bytes()).hexdigest())
            result_path.with_suffix(".json").write_text(json.dumps(metadata, indent=2))
        except Exception:
            result_path.unlink(missing_ok=True)
            if warmup_path:
                warmup_path.unlink(missing_ok=True)
            raise
    return result_path

print("Compilation terminée. CPU embarqué activé :", RUN_CPU_REFERENCE)


## 4. Test rapide
Trois runs en D=10, P=50 sur les quatre fonctions. Les optima théoriques sont
−450, −330, 390 et −180 ; cela ne signifie pas que le PSO doit tous les atteindre.
Une incohérence entre fitness GPU et recomposition CPU provoque un échec explicite.


In [ ]:
smoke_args = ["--func", "all", "--dim", 10, "--pop", 50, "--runs", 3,
              "--seed", SEED, "--shift", "rand"]
run_pso("gpu", smoke_args, "smoke_pso_gpu.csv", expected_rows=12)
if RUN_CPU_REFERENCE:
    run_pso("cpu", smoke_args, "smoke_pso_cpu_reference.csv", expected_rows=12)


## 5. Configuration libre
Fonctions : `all`, 0 Lévy, 1 Rastrigin, 2 Rosenbrock, 3 Griewank, 4 Sphere.
Pour un shift fourni par fichier, choisir une seule fonction et un chemin absolu.
`rand` est un décalage synthétique reproductible, pas une garantie de reproduction
des décalages de l'article. Cette cellule ne remplace pas les résultats du sweep.


In [ ]:
FUNC, DIM, POP, CUSTOM_RUNS = "all", 50, 100, 10
CUSTOM_SHIFT = SHIFT
custom_args = ["--func", FUNC, "--dim", DIM, "--pop", POP, "--runs", CUSTOM_RUNS,
               "--seed", SEED, "--shift", CUSTOM_SHIFT]
run_pso("gpu", custom_args)
if RUN_CPU_REFERENCE:
    run_pso("cpu", custom_args)


## 6. Protocole complet
4 fonctions × 3 dimensions × 3 populations × 10 runs = **360 lignes par implémentation**.
Le shift et la graine sont explicitement transmis aux deux exécutables.
Les CSV, journaux et métadonnées sont sauvegardés dans `pso_gpu_work/results/`.
Le CSV CPU est uniquement la référence embarquée, distincte de l'autre notebook.
Chaque configuration GPU est désormais précédée du warm-up représentatif décrit en section 8 ;
ces runs supplémentaires augmentent la durée totale, mais sont exclus des statistiques.
Les temps d'échauffement sont conservés séparément dans `*_warmup.csv` et identifiés dans le JSON.


In [ ]:
if not isinstance(RUNS, int) or RUNS < 1:
    raise ValueError("RUNS doit être un entier positif")
if SHIFT not in ("rand", "zero"):
    raise ValueError("Le sweep multi-fonctions nécessite SHIFT='rand' ou 'zero'. Utiliser la cellule libre pour file:.")
SWEEP_ARGS = ["--sweep", "--runs", RUNS, "--seed", SEED, "--shift", SHIFT]
GPU_CSV = run_pso("gpu", SWEEP_ARGS, "results_pso_gpu.csv", expected_rows=36 * RUNS)
CPU_CSV = None


In [ ]:
if RUN_CPU_REFERENCE:
    CPU_CSV = run_pso("cpu", SWEEP_ARGS, "results_pso_cpu_reference.csv", expected_rows=36 * RUNS)
else:
    CPU_CSV = None
    print("Référence CPU désactivée ; analyse GPU seule.")


## 7. Analyse des résultats de cette exécution
### 7.1 Qualité, temps et comparaison facultative à la référence CPU
Erreur = f − f*, succès si erreur < 10⁻⁸. Les fichiers lus sont exclusivement ceux
du sweep ci-dessus, avec contrôle d'empreinte et identification `impl`.
Les accélérations comparent les temps mesurés du calcul, pas les temps bout en bout.
Ce notebook ne réalise pas les tests statistiques du projet : ils seront à traiter
avec les résultats DE/PSO harmonisés (ne pas apparier uniquement sur le numéro de seed).


In [ ]:
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 200, "display.max_rows", 200)
frames = []
for csv_path in [GPU_CSV] + ([CPU_CSV] if CPU_CSV is not None else []):
    metadata = json.loads(csv_path.with_suffix(".json").read_text())
    if hashlib.sha256(csv_path.read_bytes()).hexdigest() != metadata["csv_sha256"]:
        raise ValueError(f"CSV modifié après exécution : {csv_path}")
    frame = pd.read_csv(csv_path)
    frame["impl"] = metadata["impl"]
    frames.append(frame)
df = pd.concat(frames, ignore_index=True)
df["success"] = df["error"] < 1e-8
summary = (df.groupby(["impl", "func", "dim", "pop"])
    .agg(runs=("run", "count"), mean_best=("best", "mean"), std_best=("best", "std"),
         min_best=("best", "min"), mean_error=("error", "mean"), std_error=("error", "std"),
         median_error=("error", "median"), success_rate=("success", "mean"),
         time_s=("time_ms", lambda s: s.mean() / 1000.0))
    .reset_index())
if not summary["runs"].eq(RUNS).all() or len(summary) != 36 * len(frames):
    raise ValueError("Sweep incomplet")
df.to_csv(RESULTS_DIR / "results_pso_combined.csv", index=False)
summary.to_csv(RESULTS_DIR / "summary_pso_by_impl.csv", index=False)
display(summary)

speedup = None
if CPU_CSV is not None:
    times = summary.pivot(index=["func", "dim", "pop"], columns="impl", values="time_s")
    speedup = (times["PSO cpu"] / times["PSO gpu"]).rename("speedup_cpu_over_gpu").reset_index()
    speedup.to_csv(RESULTS_DIR / "speedup_pso.csv", index=False)
    display(speedup)


### 7.2 Temps moyens par implémentation et accélération CPU/GPU


In [ ]:
import matplotlib.pyplot as plt

funcs = list(summary["func"].unique())
dims = sorted(summary["dim"].unique())
pops = sorted(summary["pop"].unique())
fig, axes = plt.subplots(len(funcs), len(dims), figsize=(4.2 * len(dims), 2.8 * len(funcs)), squeeze=False)
for i, func in enumerate(funcs):
    for j, dim in enumerate(dims):
        subset = summary[(summary.func == func) & (summary.dim == dim)]
        subset.pivot(index="pop", columns="impl", values="time_s").reindex(pops).plot.bar(ax=axes[i][j])
        axes[i][j].set(title=f"{func} — {dim}D", xlabel="population", ylabel="temps moyen (s)")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "fig_pso_time_vs_pop.png", dpi=200)
plt.show()

if speedup is not None:
    fig, axes = plt.subplots(1, len(dims), figsize=(4.5 * len(dims), 3.5), squeeze=False)
    for j, dim in enumerate(dims):
        for func in funcs:
            subset = speedup[(speedup.func == func) & (speedup.dim == dim)].sort_values("pop")
            axes[0][j].plot(subset["pop"], subset["speedup_cpu_over_gpu"], marker="o", label=func)
        axes[0][j].axhline(1, color="black", linestyle="--")
        axes[0][j].set(title=f"{dim}D", xlabel="population", ylabel="temps CPU / temps GPU")
        axes[0][j].legend()
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "fig_pso_speedup.png", dpi=200)
    plt.show()


## 8. Diagnostic ciblé : Sphere 10D P50 après échauffement intégré

**Pour refaire uniquement ce test** : exécuter la configuration/prérequis (section 1),
la génération des sources (section 2), la compilation (section 3), puis les cellules ci-dessous.
Inutile d'exécuter le smoke test, la configuration libre ou le sweep.

Chaque lancement réalise l'échauffement puis les mesures **dans le même processus CUDA**.
Avant chaque configuration GPU, au moins `WARMUP_RUNS=3` runs complets sont exécutés,
jusqu'à au moins `WARMUP_SECONDS=2.0` secondes GPU cumulées, avec un plafond de
`WARMUP_MAX_RUNS=200`. Le plafond déclenche un avertissement s'il empêche d'atteindre la cible.
Ces seuils sont un protocole d'échauffement, **pas une garantie de stabilité des fréquences**.

Même fonction, dimension, population, shift, précision et budget que les mesures.
Chaque run repart d'un essaim neuf ; les graines des runs mesurés restent `SEED + run`.
Les évaluations d'échauffement sont du travail supplémentaire, exclu du budget des runs mesurés
et de leurs statistiques. Le protocole CPU embarqué n'est pas modifié.

Trois séries séparées, utilisant les mêmes graines, sont enregistrées sans modifier les résultats du sweep ni
le fichier `sphere_10d_p50_retest.csv` précédent. Les fichiers `*_warmup.csv` contiennent
uniquement l'échauffement ; les CSV sans ce suffixe contiennent uniquement les 10 mesures.
Tous les temps mesurés sont conservés, sans sélectionner les runs les plus rapides.


In [ ]:
DIAGNOSTIC_REPEATS = 3
DIAGNOSTIC_RUNS = 10
# 'rand' reproduit le shift du test précédent, indépendamment du SHIFT du sweep.
diagnostic_args = ["--func", 4, "--dim", 10, "--pop", 50,
                   "--runs", DIAGNOSTIC_RUNS, "--seed", SEED, "--shift", "rand"]
diagnostic_paths = []
for repetition in range(1, DIAGNOSTIC_REPEATS + 1):
    print(f"\nDiagnostic Sphere 10D P50 — répétition {repetition}/{DIAGNOSTIC_REPEATS}")
    diagnostic_paths.append(run_pso(
        "gpu", diagnostic_args,
        f"sphere_10d_p50_warmup_test_{repetition:02d}.csv",
        expected_rows=DIAGNOSTIC_RUNS))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
from os.path import relpath

diagnostic_summary = []
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for repetition, csv_path in enumerate(diagnostic_paths, 1):
    metadata = json.loads(csv_path.with_suffix(".json").read_text())
    warm_path = csv_path.parent / metadata["warmup"]["csv_file"]
    for path, digest in [(csv_path, metadata["csv_sha256"]),
                         (warm_path, metadata["warmup"]["csv_sha256"])]:
        if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
            raise ValueError(f"Fichier modifié : {path}")
    measured = pd.read_csv(csv_path).sort_values("run")
    warm = pd.read_csv(warm_path).sort_values("warmup_run")
    times = measured["time_ms"]
    diagnostic_summary.append(dict(
        repetition=repetition, measured_runs=len(times), warmup_runs=len(warm),
        warmup_target_reached=all(c["target_reached"] for c in metadata["warmup"]["configurations"]),
        mean_ms=times.mean(), std_ms=times.std(ddof=1), median_ms=times.median(),
        min_ms=times.min(), max_ms=times.max(),
        first_3_mean_ms=times.iloc[:3].mean(), last_3_mean_ms=times.iloc[-3:].mean(),
        success_rate=(measured["error"] < 1e-8).mean()))
    axes[0].plot(warm["warmup_run"], warm["time_ms"], label=f"répétition {repetition}")
    axes[1].plot(measured["run"], times, marker="o", label=f"répétition {repetition}")
    display(FileLink(relpath(csv_path, Path.cwd())))
    display(FileLink(relpath(warm_path, Path.cwd())))
    display(FileLink(relpath(csv_path.with_suffix(".json"), Path.cwd())))
diagnostic_summary = pd.DataFrame(diagnostic_summary)
diagnostic_summary.to_csv(RESULTS_DIR / "sphere_10d_p50_warmup_diagnostic_summary.csv", index=False)
display(diagnostic_summary)
axes[0].set(title="Échauffement — exclu des statistiques", xlabel="run d'échauffement", ylabel="temps GPU (ms)")
axes[1].set(title="Mesures — tous les runs conservés", xlabel="run mesuré", ylabel="temps GPU (ms)")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "sphere_10d_p50_warmup_diagnostic.png", dpi=200)
plt.show()
print("Vérifier la dérive des temps et la répétabilité entre séries ; atteindre le seuil de warm-up ne prouve pas la stabilité.")


## Modifications de reproductibilité

- **Sources isolées et toujours générées** : correction de `NAMES`/`SOURCES`, suppression de la priorité aux datasets et fichiers préexistants ; évite de compiler le DE à la place du PSO.
- **Prérequis et architecture contrôlés** : erreur explicite sans GPU/toolkit/dépendances ; pas de fallback silencieux vers une architecture incorrecte.
- **Compilation et exécution vérifiées** : C++14 explicite, arguments sans shell, propagation des codes de retour ; un ancien binaire n'est pas lancé après une compilation ratée dans une exécution linéaire.
- **CLI durcie** : `--out` sans argument et `--runs 0` sont refusés ; une fitness non finie ou incohérente fait échouer le programme.
- **Résultats dédiés et vérifiés** : aucun ancien CSV racine écrasé ; contrôle du schéma, du budget, des doublons, des valeurs finies et du nombre de runs ; sources, commandes et matériel identifiés dans les métadonnées.
- **Analyse CPU/GPU explicite** : référence CPU facultative, CSV combiné identifié et graphiques comparatifs ; pas de confusion avec le PSO CPU de l'autre notebook.
- **Anciennes sorties effacées** : elles ne correspondent plus aux cellules corrigées ; elles doivent être régénérées sur GPU.
- **Échauffement représentatif intégré** : remplace le mini-test 2D/P32 par des runs de la configuration mesurée, dans le même processus, avant chaque configuration GPU. Seuils configurables (3 runs et 2 secondes GPU, plafond 200), CSV séparés et métadonnées ; la stabilité reste à vérifier.
- **Diagnostic Sphere 10D P50** : trois séries de 10 mesures, tracé des temps individuels et liens de téléchargement, sans relancer le sweep ni écraser ses résultats.

Les kernels, formules, paramètres et générateurs aléatoires PSO n'ont pas été changés.
